# Bases de données — Exercice 6 : achats, articles et clients (correction)

Ce notebook contient, pour chaque question, trois blocs **toujours séparés** :

| Bloc | Contenu |
|:---|:---|
| 📝 **ÉNONCÉ** | la question, recopiée du sujet |
| ✅ **RÉSOLUTION** | la réponse attendue (requête SQL, code et/ou texte) |
| 💡 **EXPLICATION PÉDAGOGIQUE** | pourquoi ça marche, le lien avec le cours, les erreurs à éviter. **N'est pas à recopier.** |

**Notions travaillées :** schéma relationnel, clé primaire composée de plusieurs attributs, clé étrangère, `CREATE TABLE` avec `PRIMARY KEY` et `FOREIGN KEY`, contrainte d'intégrité référentielle.


Exécuter les cellules **dans l'ordre** (*Cell → Run All*).

> On considère une base de données qui décrit des informations sur des articles et des clients qui achètent ces articles. Le schéma relationnel de cette base est le suivant, où la clé primaire de chaque relation est soulignée et les attributs qui forment une clé étrangère sont suivis de *.
>
> - Achats (<u>aid*, cid*, dateachat</u>, quantité)
> - Articles (<u>aid</u>, rayon, prix)
> - Clients (<u>cid</u>, nom, datenaiss)

---
# MISE EN PLACE

En classe, les requêtes sont tapées dans la **console H2** (un SGBD). Ici, on utilise le module `sqlite3` de Python, qui contient un autre SGBD (**SQLite**) : **le langage SQL est le même**, donc chaque requête de ce notebook peut être recopiée telle quelle dans la console H2 (les rares différences sont signalées).

- `execute_fichier("fichier.sql")` joue le rôle de `RUNSCRIPT FROM 'fichier.sql';` dans H2 ;
- `affiche(requete)` exécute une requête et affiche le résultat sous forme de tableau, comme la console H2 ;
- `resultat(requete)` renvoie la liste des enregistrements : elle ne sert qu'aux vérifications.

La requête SQL est écrite entre trois guillemets `""" ... """` : c'est une chaîne de caractères Python qui peut tenir sur plusieurs lignes.

In [1]:
import sqlite3

base = sqlite3.connect(":memory:")   # une base de données vide, en mémoire vive


def execute_fichier(nom_fichier):
    """Équivalent de RUNSCRIPT FROM '...' dans H2 : exécute un fichier .sql."""
    fichier = open(nom_fichier, encoding="utf-8")
    script = fichier.read()
    fichier.close()
    base.executescript(script)


def resultat(requete):
    """Renvoie la liste des enregistrements obtenus (pour les vérifications)."""
    return base.execute(requete).fetchall()


def affiche(requete, maximum=20):
    """Exécute une requête et affiche le résultat comme dans la console H2."""
    curseur = base.execute(requete)
    if curseur.description is None:          # INSERT, UPDATE, DELETE, CREATE...
        print("Instruction exécutée.")
        return
    entete = ""
    for colonne in curseur.description:
        entete = entete + colonne[0] + " | "
    print(entete)
    nombre = 0
    for ligne in curseur:
        if nombre < maximum:
            texte = ""
            for valeur in ligne:
                texte = texte + str(valeur) + " | "
            print(texte)
        nombre = nombre + 1
    if nombre > maximum:
        print("... (seules les " + str(maximum) + " premières lignes sont affichées)")
    print("(" + str(nombre) + " ligne(s))")


base.execute("PRAGMA foreign_keys = ON")   # SQLite : activer la vérification des clés étrangères (H2 le fait toujours)

---
# QUESTION 1

#### 📝 ÉNONCÉ

> 1. Est-il possible pour un client d'acheter un même article plusieurs fois ?

#### ✅ RÉSOLUTION

**Oui, mais à des dates différentes.**

La clé primaire de la relation `Achats` est le triplet (`aid`, `cid`, `dateachat`). Deux achats ne peuvent pas avoir le même triplet. Un même client (`cid`) peut donc acheter un même article (`aid`) plusieurs fois, **à condition que la date soit différente** à chaque fois.

En revanche, il ne peut pas y avoir deux enregistrements pour le même client, le même article et la même date. S'il achète deux fois le même article le même jour, on l'enregistre en une seule ligne, avec l'attribut `quantité`.

#### 💡 EXPLICATION PÉDAGOGIQUE

- **Lire un schéma :** les attributs soulignés forment **ensemble** la clé primaire. Ici, ce n'est ni `aid` seul, ni `cid` seul, mais le **groupe des trois**.
- Si la clé primaire était seulement (`aid`, `cid`), un client ne pourrait acheter un article **qu'une seule fois dans toute sa vie** : ce serait absurde pour un magasin.
- L'attribut `quantité` répond au cas « plusieurs exemplaires le même jour ».
- `aid*` et `cid*` sont aussi des **clés étrangères** : `aid` fait référence à un article de la relation `Articles`, `cid` à un client de la relation `Clients`. Un attribut peut être à la fois dans la clé primaire et une clé étrangère.

---
# QUESTION 2

#### 📝 ÉNONCÉ

> 2. Quelles sont les instructions SQL permettant de créer les relations ci-dessus avec H2 ?

#### ✅ RÉSOLUTION

On crée d'abord les deux relations référencées, puis `Achats`, qui fait référence aux deux autres :

```sql
CREATE TABLE Articles (
    aid INT,
    rayon VARCHAR,
    prix REAL,
    PRIMARY KEY (aid) );

CREATE TABLE Clients (
    cid INT,
    nom VARCHAR,
    datenaiss DATE,
    PRIMARY KEY (cid) );

CREATE TABLE Achats (
    aid INT,
    cid INT,
    dateachat DATE,
    quantité INT,
    PRIMARY KEY (aid, cid, dateachat),
    FOREIGN KEY (aid) REFERENCES Articles(aid),
    FOREIGN KEY (cid) REFERENCES Clients(cid) );
```

*Exécution dans le notebook :*

In [2]:
affiche("""
CREATE TABLE Articles (
    aid INT,
    rayon VARCHAR,
    prix REAL,
    PRIMARY KEY (aid) );
""")
affiche("""
CREATE TABLE Clients (
    cid INT,
    nom VARCHAR,
    datenaiss DATE,
    PRIMARY KEY (cid) );
""")
affiche("""
CREATE TABLE Achats (
    aid INT,
    cid INT,
    dateachat DATE,
    quantité INT,
    PRIMARY KEY (aid, cid, dateachat),
    FOREIGN KEY (aid) REFERENCES Articles(aid),
    FOREIGN KEY (cid) REFERENCES Clients(cid) );
""")

Instruction exécutée.
Instruction exécutée.
Instruction exécutée.


#### 💡 EXPLICATION PÉDAGOGIQUE

- **Choisir les types :** les identifiants sont des entiers (`INT`), les textes des `VARCHAR`, le prix un nombre à virgule (`REAL`), les dates des `DATE` (en SQL, une date s'écrit `'AAAA-MM-JJ'`).
- **Clé primaire :** `PRIMARY KEY (aid)` ; pour une clé de plusieurs attributs, on les liste **tous dans les mêmes parenthèses** : `PRIMARY KEY (aid, cid, dateachat)`. ⚠️ Écrire trois fois `PRIMARY KEY` serait une erreur : une relation n'a qu'**une** clé primaire.
- **Clés étrangères :** `FOREIGN KEY (aid) REFERENCES Articles(aid)` signifie « la valeur de `aid` dans `Achats` doit exister dans l'attribut `aid` de `Articles` ».
- ⚠️ **L'ordre de création compte :** `Achats` fait référence à `Articles` et `Clients`, qui doivent donc **déjà exister**. Créer `Achats` en premier provoque une erreur dans H2 (« table ARTICLES non trouvée »).
- L'accent de `quantité` est accepté par H2 et SQLite ; on peut aussi écrire `quantite` pour éviter tout souci.

---
# QUESTION 3

#### 📝 ÉNONCÉ

> 3. On dispose de trois fichiers achats.sql, articles.sql et clients.sql qui permettent d'effectuer des enregistrements dans chacune des relations ci-dessus.
> On utilise la commande RUNSCRIPT pour exécuter les instructions SQL contenues dans ces fichiers : quel ordre chronologique peut-on choisir pour alimenter la base de données avec les fichiers mis à disposition ?

#### ✅ RÉSOLUTION

Il faut exécuter `articles.sql` et `clients.sql` **avant** `achats.sql` (l'ordre entre `articles.sql` et `clients.sql` n'a pas d'importance) :

```sql
RUNSCRIPT FROM 'articles.sql';
RUNSCRIPT FROM 'clients.sql';
RUNSCRIPT FROM 'achats.sql';
```

ou bien `clients.sql`, puis `articles.sql`, puis `achats.sql`.

**Justification :** chaque achat fait référence à un article (`aid`) et à un client (`cid`) par ses clés étrangères. À cause de la **contrainte d'intégrité référentielle**, le SGBD refuse d'enregistrer un achat dont l'article ou le client n'existe pas encore.

*Illustration (non demandée), avec des données inventées :*

In [3]:
# Achat d'un client qui n'existe pas encore : refusé
try:
    affiche("INSERT INTO Achats VALUES (10, 1, '2026-11-14', 2)")
except sqlite3.IntegrityError as erreur:
    print("Refusé par le SGBD :", erreur)

# Dans le bon ordre : article et client d'abord, puis l'achat
affiche("INSERT INTO Articles VALUES (10, 'Papeterie', 2.5)")
affiche("INSERT INTO Clients VALUES (1, 'Durand', '1990-05-17')")
affiche("INSERT INTO Achats VALUES (10, 1, '2026-11-14', 2)")
affiche("INSERT INTO Achats VALUES (10, 1, '2026-11-20', 1)")   # même article, autre date : accepté
affiche("SELECT * FROM Achats")

# Même client, même article, même date : refusé (question 1)
try:
    affiche("INSERT INTO Achats VALUES (10, 1, '2026-11-14', 5)")
except sqlite3.IntegrityError as erreur:
    print("Refusé par le SGBD :", erreur)

assert len(resultat("SELECT * FROM Achats")) == 2

Refusé par le SGBD : FOREIGN KEY constraint failed
Instruction exécutée.
Instruction exécutée.
Instruction exécutée.
Instruction exécutée.
aid | cid | dateachat | quantité | 
10 | 1 | 2026-11-14 | 2 | 
10 | 1 | 2026-11-20 | 1 | 
(2 ligne(s))
Refusé par le SGBD : UNIQUE constraint failed: Achats.aid, Achats.cid, Achats.dateachat


#### 💡 EXPLICATION PÉDAGOGIQUE

- **Contrainte d'intégrité référentielle (cours, page 6) :** une clé étrangère doit toujours avoir pour valeur une valeur **qui existe** dans l'attribut référencé. Un achat ne peut pas concerner un client « fantôme ».
- **Règle pratique :** on remplit d'abord les relations **référencées** (celles vers lesquelles pointent les flèches des clés étrangères), puis celles qui y font référence. C'est le même ordre que pour les `CREATE TABLE`.
- Pour **supprimer**, c'est l'inverse : on vide `Achats` avant `Articles` et `Clients`, sinon le SGBD refuse de supprimer un client qui a encore des achats.
- L'illustration montre aussi la question 1 : un deuxième achat du même article le même jour est refusé (contrainte de clé primaire), mais un autre jour il est accepté.
- **Différence H2 / SQLite :** H2 vérifie toujours les clés étrangères ; SQLite ne le fait que si on l'active avec `PRAGMA foreign_keys = ON` (fait dans la mise en place).